# Comparing two revisions of a pipeline

No dedicated "git diff" experiment type — `Scenario.revision` on the same
`ExperimentDef` already checks out another commit, imports the pipeline from
there, and runs it next to the baseline. `example_projects/experiments/credit_pipeline.py`
has two real revisions to compare: `HEAD` added a flat $50 affordability fee
to the ratio calculation; `HEAD^` is the commit just before it.

In [1]:
import polars as pl
from decider.experiments import Experiment, OutcomeCounts, RunStatus
from example_projects.experiments.credit_pipeline import build

ENTRY = "example_projects.experiments.credit_pipeline:build"
pipeline = build()
df = pl.DataFrame({
    "income": [1000.0, 2000.0, 1500.0, 800.0],
    "debt": [350.0, 900.0, 700.0, 300.0],
})
df

shape: (4, 2)
┌────────┬───────┐
│ income ┆ debt  │
│ ---    ┆ ---   │
│ f64    ┆ f64   │
╞════════╪═══════╡
│ 1000.0 ┆ 350.0 │
│ 2000.0 ┆ 900.0 │
│ 1500.0 ┆ 700.0 │
│ 800.0  ┆ 300.0 │
└────────┴───────┘


In [2]:
exp = (Experiment("version-drift", pipeline, df, entry=ENTRY)
       .scenario("previous_revision", revision="HEAD^")
       .plot(OutcomeCounts(type="outcome_counts", column="approved"))
       .compare(outputs=("approved",)))

result = exp.run()
assert result.status is RunStatus.COMPLETED
[(s.name, s.status.value, s.revision.resolved[:7], len(s.divergences)) for s in result.scenarios]

[('previous_revision', 'completed', '59b9825', 1)]


One record's `approved` flips once the fee is added — `result.plots` runs
over the baseline (the current code), so it shows today's answer; the
scenario's `divergences` is what actually changed against the commit before.

In [3]:
result.plots

{'outcome_counts': {'kind': 'bar', 'column': 'approved', 'series': {'False': 4}}}


One gotcha worth knowing: `Experiment(...)` needs the live pipeline object up
front, so its module is already imported in this process by the time a
`revision` scenario wants to load the *same* module from a temp checkout of
another commit. Python caches imports by name, so naively re-importing it in
this process would silently hand back the already-imported (current) code
instead of the checked-out one — at whatever depth the entry module's own
imports go, not just the one name. `decider` runs a revision scenario in a
fresh subprocess instead, specifically so this comparison is trustworthy; the
subprocess still uses this process's installed `decider` engine, only the
flow module itself loads from the checked-out tree.